# Día 3 · Cuaderno 7 — Archivos y manejo de errores

**Programar para enseñar — Python e IA generativa para Humanidades Digitales**
Formación docente para EH1023 · Pontificia Universidad Javeriana, Facultad de Ingeniería

*(Primer bloque del Día 3 — 2 horas)*

---

Hasta ahora los datos vivían dentro del programa. Hoy aprenderás a que tu programa **lea y escriba
archivos**, que es como la información sale y entra del mundo real. Y aprenderás a **manejar errores**
para que un tropiezo no detenga todo. Al terminar podrás:

- **Escribir** un archivo de texto y volver a **leerlo**.
- Usar la forma segura `with open(...)`.
- Leer **datos del repositorio** y procesarlos línea por línea.
- Capturar errores con `try` / `except` para que tu programa no se rompa.


## 1. Escribir un archivo

`open(nombre, "w")` abre un archivo para **escribir** (`w` = *write*). Si no existe, lo crea.
Después de escribir, hay que **cerrarlo** con `.close()`.

> ⚠️ El modo `"w"` **borra** el contenido anterior del archivo. Para *agregar* sin borrar se usa `"a"`.


In [ ]:
archivo = open("notas.txt", "w")
archivo.write("Primera linea: hola archivo.\n")
archivo.write("Segunda linea: esto es humanidades digitales.\n")
archivo.close()

print("Archivo 'notas.txt' creado y guardado.")

## 2. Leer un archivo

`open(nombre, "r")` lo abre para **leer** (`r` = *read*). Con `.read()` obtenemos todo el contenido
como una sola cadena.


In [ ]:
archivo = open("notas.txt", "r")
contenido = archivo.read()
archivo.close()

print(contenido)

También podemos recorrer un archivo **línea por línea** con un `for`:


In [ ]:
archivo = open("notas.txt", "r")
for linea in archivo:
    print("Linea leida:", linea.strip())   # .strip() quita el salto de linea
archivo.close()

## 3. La forma recomendada: `with`

Es fácil olvidar el `.close()`. Por eso, la forma profesional y segura es `with`, que **cierra el
archivo automáticamente** al terminar el bloque.


In [ ]:
with open("notas.txt", "r") as archivo:
    contenido = archivo.read()

print(contenido)
print("(El archivo ya se cerro solo.)")

> 🧭 **Nota para docentes.** En Colab, los archivos que creas se guardan en la sesión y desaparecen
> al cerrarla. Conviene advertirlo a tus estudiantes para que no se asusten si "se pierde" un archivo:
> no es un error, es que la sesión es temporal. Para conservarlo, **Archivo → Descargar**.


## 4. Leer datos del repositorio y procesarlos

Traemos el archivo de fechas del repositorio y lo procesamos **línea por línea**. Cada línea tiene la
forma `evento,anio`. Solo **ejecuta** la primera celda (descarga los datos).


In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/fechas_historicas.csv"
texto = urllib.request.urlopen(url).read().decode("utf-8")
print(texto)

Ahora separamos por líneas con `.splitlines()`, saltamos el encabezado y partimos cada línea por la
coma con `.split(",")`:


In [ ]:
lineas = texto.splitlines()

for linea in lineas[1:]:            # [1:] omite el encabezado "evento,anio"
    partes = linea.split(",")        # ["Fundacion de Tenochtitlan", "1325"]
    evento = partes[0]
    anio = int(partes[1])            # convertimos el anio de texto a numero
    print(f"{anio} -> {evento}")

## 5. Manejar errores: `try` / `except`

A veces algo falla: un archivo no existe, un texto no se puede convertir a número, pedimos una
posición que no está... Sin protección, el programa **se detiene** con un mensaje rojo. La buena
noticia: ese mensaje es una **pista**. Vamos a hacerlo en tres pasos: **(1)** provocar los errores y
leerlos, **(2)** capturarlos con `try`/`except`, y **(3)** aprender a atrapar incluso un error que **no
esperábamos**.

### Paso 1 — Provoca los errores y obsérvalos

Ejecuta **cada celda por separado**. Verás un mensaje rojo: **lee siempre la última línea**, que dice
el **nombre** del error. No te asustes, eso es lo que queremos ver aquí.

In [ ]:
# (1) PRODUCE un error a proposito: el archivo no existe.
# Lee la ULTIMA linea: dira "FileNotFoundError".
with open("archivo_que_no_existe.txt", "r") as f:
    datos = f.read()

In [ ]:
# (3) PRODUCE un error a proposito: pedimos una posicion que no existe en la lista.
# Ultima linea: "IndexError".
autores = ["Cervantes", "Borges"]
print(autores[5])

In [ ]:
# (4) PRODUCE un error a proposito: dividir entre cero.
# Ultima linea: "ZeroDivisionError".
total_palabras = 100
numero_de_textos = 0
print("Promedio:", total_palabras / numero_de_textos)

### Paso 2 — Captúralos con `try` / `except`

Ahora protegemos el código: ponemos lo riesgoso en `try` y, si falla, respondemos en `except` con un
mensaje claro. El programa **ya no se detiene**.

In [ ]:
try:
    with open("archivo_que_no_existe.txt", "r") as f:
        datos = f.read()
except FileNotFoundError:
    print("No encontre el archivo. Revisa el nombre o la ruta.")

Otro caso clásico: convertir a número algo que no lo es.


In [ ]:
dato = "mil seiscientos cinco"   # esto NO es un numero

try:
    anio = int(dato)
    print("El anio es:", anio)
except ValueError:
    print(f"'{dato}' no es un numero valido. Se esperaba algo como 1605.")

Cada tipo de error tiene su propio `except`. Aquí atrapamos el de la lista (`IndexError`):

In [ ]:
autores = ["Cervantes", "Borges"]

try:
    print(autores[5])
except IndexError:
    print("Esa posicion no existe en la lista. Revisa el indice.")

### Paso 3 — ¿Y si no sé qué error va a ocurrir?

A veces **no sabemos de antemano** qué puede fallar. Para esos casos existe `except Exception as e`, que
atrapa **cualquier** error y nos deja inspeccionarlo: `type(e).__name__` nos dice **qué tipo** fue y
`e` nos da el detalle. Así el programa sigue, aunque aparezca un error que no habíamos previsto.

En el siguiente ejemplo recorremos varios valores; uno provoca `ZeroDivisionError` y otro un error
**distinto que no anticipamos** (`TypeError`, por dividir entre un texto). Un solo `except Exception`
los maneja a ambos.

In [1]:
def dividir(a, b):
    return a / b

# El ultimo valor ("tres") provocara un error que NO esperabamos.
valores = [10, 0, "tres"]

for v in valores:
    try:
        print(f"100 / {v} = {dividir(100, v)}")
    except Exception as e:
        # type(e).__name__ revela QUE error fue, sin que tengamos que adivinarlo de antemano
        print(f"No se pudo con {v}: {type(e).__name__} -> {e}")

100 / 10 = 10.0
No se pudo con 0: ZeroDivisionError -> division by zero
No se pudo con tres: TypeError -> unsupported operand type(s) for /: 'int' and 'str'


> ⚠️ Usa `except Exception` como **red de seguridad** para lo inesperado, pero cuando **sí sabes**
> qué puede fallar, prefiere el `except` específico (`FileNotFoundError`, `ValueError`...): da mensajes
> más útiles. Lo ideal es combinar ambos.

> 🧭 **Nota para docentes.** Los mensajes de error de Python asustan al principio, pero son
> **pistas**, no castigos. Enseñar a *leer* la última línea del error (`FileNotFoundError`,
> `ValueError`, etc.) es una de las habilidades más valiosas y, de nuevo, evita la frustración:
> el estudiante deja de sentir que "se rompió todo" y entiende qué corregir.


## 6. Todo junto: leer, procesar y guardar un resultado

Combinamos lo de hoy: leemos el fragmento literario del repositorio, contamos sus palabras y
**guardamos** un pequeño reporte en un archivo nuevo.


In [4]:
import urllib.request

url = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/fragmento_literario.txt"
texto = urllib.request.urlopen(url).read().decode("utf-8")

num_palabras = len(texto.split())

with open("reporte.md", "w", encoding='utf-8') as salida:
    salida.write("# Reporte del fragmento literario\n")
    salida.write(f"Numero de palabras: **{num_palabras}**\n")

# Lo leemos de vuelta para confirmar que se guardo bien:
with open("reporte.md", "r") as f:
    print(f.read())

# Reporte del fragmento literario
Numero de palabras: **99**



### Otro ejemplo — una **función** que analiza un texto y guarda un reporte

Este ejemplo reúne lo de la semana: una **función**, un **bucle**, **condicionales** y un **diccionario**
como reporte, además de leer y escribir **archivos**.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/fragmento_literario.txt"
texto = urllib.request.urlopen(url).read().decode("utf-8")

def analizar(texto):
    palabras = texto.split()
    reporte = {"total": 0, "largas": 0, "mas_larga": ""}
    for palabra in palabras:
        reporte["total"] = reporte["total"] + 1
        if len(palabra) > 6:                       # condicional 1: contar largas
            reporte["largas"] = reporte["largas"] + 1
        if len(palabra) > len(reporte["mas_larga"]):  # condicional 2: guardar la mas larga
            reporte["mas_larga"] = palabra
    return reporte

reporte = analizar(texto)

# Guardamos el diccionario, una linea por entrada:
with open("reporte_detallado.txt", "w") as f:
    for clave, valor in reporte.items():
        f.write(f"{clave}: {valor}\n")

# Lo leemos de vuelta:
with open("reporte_detallado.txt", "r") as f:
    print(f.read())

### Otro ejemplo — clasificar eventos por siglo y guardarlos

Ahora con datos tabulares: leemos `fechas_historicas.csv`, usamos una **función** para calcular el siglo,
un **bucle** con **condicional** para armar una **lista de diccionarios**, y guardamos el resultado en un
archivo.

In [ ]:
import urllib.request

url = "https://raw.githubusercontent.com/calderonf/curso-python-humanidades-digitales/main/datos/fechas_historicas.csv"
texto = urllib.request.urlopen(url).read().decode("utf-8")
lineas = texto.splitlines()

def siglo_de(anio):
    return (anio - 1) // 100 + 1

clasificados = []
for linea in lineas[1:]:                 # saltamos el encabezado
    partes = linea.split(",")
    evento = partes[0]
    anio = int(partes[1])
    era = "d.C." if anio > 0 else "a.C."  # condicional en una linea
    clasificados.append({"evento": evento, "anio": anio, "siglo": siglo_de(anio), "era": era})

with open("eventos_por_siglo.txt", "w") as f:
    for item in clasificados:
        f.write(f"Siglo {item['siglo']} ({item['era']}): {item['evento']} - {item['anio']}\n")

with open("eventos_por_siglo.txt", "r") as f:
    print(f.read())

## 7. Ejercicios graduados

Avanza a tu ritmo. **No es obligatorio llegar al reto.**

**Nivel 1 (base).** Crea un archivo `mi_glosario.txt` y escribe en él dos líneas, cada una con un
término y su definición. Luego ábrelo y muéstralo en pantalla.

**Nivel 2.** Usa `try` / `except` para intentar convertir a número la palabra `"siglo"`. Si falla,
imprime un mensaje claro en lugar de dejar que el programa se rompa.

**Reto (opcional).** Reutiliza la lista `lineas` de la sección 4 y cuenta cuántos eventos ocurrieron
**después del año 1500**, manejando con `try`/`except` cualquier línea que no tenga un año válido.


In [ ]:
# Tu solucion del Nivel 1:


<details>
<summary>👀 Solución posible del Nivel 1</summary>

```python
with open("mi_glosario.txt", "w") as f:
    f.write("algoritmo: pasos para resolver un problema\n")
    f.write("variable: un nombre que guarda un valor\n")

with open("mi_glosario.txt", "r") as f:
    print(f.read())
```
</details>


In [ ]:
# Tu solucion del Nivel 2:


<details>
<summary>👀 Solución posible del Nivel 2</summary>

```python
try:
    numero = int("siglo")
    print(numero)
except ValueError:
    print("'siglo' no se puede convertir a numero.")
```
</details>


In [ ]:
# Tu solucion del Reto (opcional):


<details>
<summary>👀 Solución posible del Reto</summary>

```python
despues_1500 = 0
for linea in lineas[1:]:
    partes = linea.split(",")
    try:
        anio = int(partes[1])
        if anio > 1500:
            despues_1500 = despues_1500 + 1
    except (ValueError, IndexError):
        print("Linea ignorada:", linea)
print(f"Eventos despues de 1500: {despues_1500}")
```
</details>


---

## Cierre del bloque

Ahora tu programa puede **leer y escribir archivos** y **sobrevivir a los errores** con `try`/`except`.
Esto es justo lo que necesitamos para el siguiente paso: trabajar con **tablas de datos** de verdad.

### Esta tarde (segundo bloque del Día 3)

Daremos el salto a **pandas** y **matplotlib** para **manipular y visualizar datos**: cargar una tabla
de obras literarias, explorarla y convertirla en **gráficos** que cuenten algo.

> Guarda tu trabajo con **Archivo → Guardar una copia en Drive**.
